In [59]:
from gerrychain import Graph, Partition, MarkovChain
from pathlib import Path
import geopandas as gpd
import networkx as nx
from gerrychain.updaters import Tally
from functools import partial
from gerrychain.proposals import recom 
from gerrychain.accept import always_accept
from gerrychain.constraints import contiguous
import gzip 
from tqdm import tqdm
import jsonlines as jl

In [60]:
# turnout rate 
#6600 fix size of profile to size of district 
# 6600 
# 5-6 hours run tonight 

# how to decide the number of voters per district 
# you can look at historical turnout data and use that to estimate the turnout rate for each district, then use that to calculate the number of voters needed in each district to achieve a certain level of representation.\

# "Use alpha = 1 (all bets are off) for candidate preferences and turnout proportionate to VAP (in the shapefile / dualgraph)"


The total number of people in Jones county is 10153. We are spliting the county in 6 districts, 1 district, and 3 districts.
We need a total number of voters that is divisible by all those districting plans so there are a fixed pool of voters evenly divided among districts (that should be really close to ideal size)
This allows for comparitablity across districts as a win for x number of ballots in a preference profile may mean something different statistically than a y number of ballots in a preference profile. Randomness?
With a voter turnout of 
What happens with a non-uniform turnout across slates? ElectoralRedistricting has a a:0.7 AND b:0.9

We're going to assume a even 0.7 turnout per party. So we can scale the total population by that and then round to a number that is divisible by all paritions of districts 
total_seats = 6


In [61]:
NUM_DISTRICTS = 6 
NUM_WINNERS_PER_DISTRICT = 1 
POP_TOLERANCE = 0.03 # 3% population deviation tolerance 
CHAIN_LENGTH = 1000 # Length of the Markov chain
TOTAL_POPULATION = 6600 # Total population of the area being redistricted
# Shape file Column names
POP_COL = "TOTPOP"
VAP_COL ="VAP" # Voting Age Population
BVAP_COL = "BVAP" # Black Voting Age Population

RUN_NAME = "attempt_2" # Name for this run, used in output file naming

In [63]:
# Read in the jones shape file
gdf = gpd.read_file("jones_blocks.shp")
print(gdf.columns)

Index(['STATEFP10', 'COUNTYFP10', 'TRACTCE10', 'BLOCKCE10', 'GEOID10',
       'NAME10', 'MTFCC10', 'UR10', 'UACE10', 'UATYP10', 'FUNCSTAT10',
       'ALAND10', 'AWATER10', 'INTPTLAT10', 'INTPTLON10', 'TOTPOP', 'NH_WHITE',
       'NH_BLACK', 'NH_AMIN', 'NH_ASIAN', 'NH_NHPI', 'NH_OTHER', 'NH_2MORE',
       'HISP', 'H_WHITE', 'H_BLACK', 'H_AMIN', 'H_ASIAN', 'H_NHPI', 'H_OTHER',
       'H_2MORE', 'VAP', 'HVAP', 'WVAP', 'BVAP', 'AMINVAP', 'ASIANVAP',
       'NHPIVAP', 'OTHERVAP', '2MOREVAP', 'NAME', 'state', 'county', 'tract',
       'block', 'geoid', 'BGID', 'CVAP', 'WCVAP', 'BCVAP', 'HCVAP', 'AMINCVAP',
       'ASIANCVAP', 'NHPICVAP', 'geometry'],
      dtype='str')


In [64]:
print(f"{len(gdf)} census blocks/precincts")

941 census blocks/precincts


In [65]:
# Fill any missing demographic values with 0
for col in [POP_COL, VAP_COL, BVAP_COL]:
    gdf[col] = gdf[col].fillna(0).astype(int)

# County-level summary
total_pop  = gdf[POP_COL].sum()
total_vap  = gdf[VAP_COL].sum()
total_bvap = gdf[BVAP_COL].sum()
zero_pop   = (gdf[POP_COL] == 0).sum()

print(f"    Total population  : {total_pop}")
print(f"    Voting age pop    : {total_vap}")
print(f"    Black VAP (BVAP)  : {total_bvap}  ({100*total_bvap/total_vap:.1f}% of VAP)")
print(f"    Non-Black VAP     : {total_vap-total_bvap}  ({100*(total_vap-total_bvap)/total_vap:1f}% of VAP)")
print(f"    Ideal district pop: {total_pop/NUM_DISTRICTS:.0f} ({NUM_DISTRICTS} Districts)")
print(f"\n    Zero-population blocks: {zero_pop} of {len(gdf)} ({100*zero_pop/len(gdf):1f}%)")

    Total population  : 10153
    Voting age pop    : 7945
    Black VAP (BVAP)  : 2524  (31.8% of VAP)
    Non-Black VAP     : 5421  (68.231592% of VAP)
    Ideal district pop: 1692 (6 Districts)

    Zero-population blocks: 524 of 941 (55.685441%)


In [ ]:
# build the dual graph 
graph = Graph.from_geodataframe(gdf)
print(graph)

Graph with 941 nodes and 2227 edges


/Users/gracegibson/work/jones_project/.venv/lib/python3.11/site-packages/gerrychain/graph/graph.py:266: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  areas = df.geometry.area.to_dict()
/Users/gracegibson/work/jones_project/.venv/lib/python3.11/site-packages/gerrychain/graph/graph.py:457: UserWarning: NA values found in column UACE10!
  warnings.warn("NA values found in column {}!".format(column))
/Users/gracegibson/work/jones_project/.venv/lib/python3.11/site-packages/gerrychain/graph/graph.py:457: UserWarning: NA values found in column UATYP10!
  warnings.warn("NA values found in column {}!".format(column))


In [ ]:
# relabel nodes as 0-indexed integers for list-based assignment serialization 
graph = Graph.from_networkx(nx.convert_node_labels_to_integers(graph, first_label=0))
print(graph)

Graph with 941 nodes and 2227 edges


In [ ]:
district_nums = [int(d["num_districts"]) for d in districting_config]
output_dir = Path(f"outputs/{RUN_NAME}/districts")
graph_node_order = list(graph.nodes)
output_dir.mkdir(parents=True, exist_ok=True)
for num_districts in district_nums:
    if num_districts == 1:
        print("Only one district, skipping Markov chain generation since no districting plan yet")
        assignment = [0] * len(graph_node_order)
        output_path = output_dir / f"{RUN_NAME}_{num_districts}_districts.jsonl.gz"
        print(f"Saving districting plans to: {output_path}")
        with gzip.open(output_path, mode="wt", encoding="utf-8") as gz_file:
            writer = jl.Writer(gz_file)
            writer.write({"assignment": assignment, "sample": 1,})
        continue
    # create intital partition of graph (initial state of Markov chain)
    my_updaters = {
        "population": Tally(POP_COL),
        "vap": Tally(VAP_COL),
        "bvap": Tally(BVAP_COL),
    }
    partition = Partition.from_random_assignment(
        graph = graph,
        n_parts = num_districts,
        epsilon = POP_TOLERANCE,
        pop_col = POP_COL,
        updaters = my_updaters
    )
    ideal_pop = sum(partition["population"].values()) / num_districts
    # run the markov chain with the Recom proposal (merge 2 adjacent districts and re-split with a spanning tree)
    proposal = partial(
        recom,
        pop_col = POP_COL,
        pop_target = ideal_pop,
        epsilon = POP_TOLERANCE,
    )
    chain = MarkovChain(
        proposal=proposal,
        constraints=[],
        accept=always_accept,
        initial_state=partition,
        total_steps=CHAIN_LENGTH,
    )
    output_path = output_dir / f"{RUN_NAME}_{num_districts}_districts.jsonl.gz"
    print(f"Saving districting plans to: {output_path}")
    with gzip.open(output_path, mode="wt", encoding="utf-8") as gz_file:
        writer = jl.Writer(gz_file)
        for sample_num, step in enumerate(tqdm(chain, total=CHAIN_LENGTH, desc=f"Generating {NUM_DISTRICTS}-district plans"), start=1):
            assignment = list(step.assignment.to_series().loc[graph_node_order].astype(int))
            # add demographic information
            # print(f"Step {sample_num}:")
            # print(step.keys())
            # print("Population per district:")
            # print(step["population"])

            # demographics = {
            #     str(d) :
            #     {
            #         "population": step["population"][d],
            #         "vap": step["vap"][d],
            #         "bvap": step["bvap"][d]
            #     }
            #     for d in sorted(step.parts.keys())
            # }
            writer.write({"assignment": assignment, "sample": sample_num,})
        writer.close()

Saving districting plans to: outputs/attempt_2/districts/attempt_2_6_districts.jsonl.gz


Generating 6-district plans: 100%|██████████| 1000/1000 [00:18<00:00, 53.39it/s]


Saving districting plans to: outputs/attempt_2/districts/attempt_2_3_districts.jsonl.gz


Generating 6-district plans: 100%|██████████| 1000/1000 [00:23<00:00, 42.23it/s]
/Users/gracegibson/work/jones_project/.venv/lib/python3.11/site-packages/gerrychain/tree.py:704: BipartitionWarning: 
Failed to find a balanced cut after 1000 attempts.
If possible, consider enabling pair reselection within your
MarkovChain proposal method to allow the algorithm to select
a different pair of districts for recombination.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
print(f"Graph node order: {graph_node_order}")

Graph node order: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 2

In [ ]:
for sample_num, step in enumerate(chain):
    print(step.assignment.to_series())
    break

11     0
12     0
13     0
14     0
559    0
      ..
243    5
244    5
252    5
253    5
254    5
Length: 941, dtype: int64


In [ ]:
from votekit.ballot_generator import BlocSlateConfig, slate_pl_profile_generator
import pandas as pd
import json

In [80]:
NUM_SLATES = 2
num_candidates_per_slate = [3,4,8]
ALPHA = 1.0

# Cohesion values
cohesion_values = {
    "B": {
        "B": 0.98,
        "NB": 0.02,
        },
    "NB": {
        "B": 0.13,
        "NB": 0.87,
        }
    }
alpha_values = {
    "B": {
        "B": ALPHA,
        "NB": ALPHA,
        },
    "NB": {
        "B": ALPHA,
        "NB": ALPHA,
        }
    }
print(num_candidates_per_slate[0])

3


In [96]:
# preference profile generation (generate a preference profile for each district using s-PL)

# need to define a bloc slate config per district (based on the district's BVAP and VAP)
# generate settings for each sampled district plan (compute per-district bloc proportions based on BVAP/VAP
gdf = gpd.read_file("jones_blocks.shp")
demo_df = gdf[[POP_COL, VAP_COL, BVAP_COL]].copy()
for config_num, num_districts in enumerate(district_nums):
    path_to_districting = f"outputs/{RUN_NAME}/districts/{RUN_NAME}_{num_districts}_districts.jsonl.gz"
    all_plans = []
    if num_districts == 1:
        print("Warning: Only one district, skipping preference profile generation for now since no districting plan yet")
        break
    with gzip.open(path_to_districting, mode="rt", encoding="utf-8") as gz_file:
        reader = jl.Reader(gz_file)
        for plan in reader:
            all_plans.append(plan)
    settings_folder = Path(f"outputs/{RUN_NAME}/settings/{num_districts}_districts")
    settings_folder.mkdir(parents=True, exist_ok=True)
    for plan_idx, plan in enumerate(all_plans):
        assignment = pd.Series(plan["assignment"], index=graph_node_order)
        demo_df["district"] = assignment # district assignment per node 
        district_demos = demo_df.groupby("district")[[POP_COL, VAP_COL, BVAP_COL]].sum()
        
        for district_label, row in district_demos.iterrows():

            # make settings
            vap = int(row[VAP_COL])
            bvap = int(row[BVAP_COL])
            nbvap = vap - bvap
            bvap_prop = round(bvap / vap, 2) if vap > 0 else 0
            nbvap_prop = 1.0 - bvap_prop
            if (bvap_prop + nbvap_prop) > 1.0 :
                print(f"District {district_label}: BVAP% = {bvap_prop:.2f}, NBVAP% = {nbvap_prop:.2f}") 
            black_candidates = []
            nonblack_candidates = []
            for cand_num in range(num_candidates_per_slate[config_num]):

                black_candidates.append("B" + str(cand_num+1))
                nonblack_candidates.append("NB" + str(cand_num+1))

            settings = {
                "district" : str(district_label),
                "sample" : str(plan["sample"]),
                "bloc_proportions" : {
                    "B": bvap_prop,
                    "NB": nbvap_prop
                },
                "slate_to_candidates" : {
                    "B" : black_candidates,
                    "NB" : nonblack_candidates
                },
                "cohesion_parameters" : cohesion_values,
                "bvap" : str(int(row[BVAP_COL])),
                "vap" : str(int(row[VAP_COL])),
                "alphas" : alpha_values,
                "num_voters" : TOTAL_POPULATION // num_districts,
                "total_seats" : 6, 
                "chain_length" : CHAIN_LENGTH,
                
            }
            out_path = settings_folder / f"{RUN_NAME}_{num_districts}_districts_settings_sample_{plan['sample']}_district_{district_label+1}.json"
            with open(out_path, "w") as f:
                json.dump(settings, f, indent=4)



In [97]:
# process settings files into Bloc Slate configurations 
from votekit.ballot_generator import (
    BlocSlateConfig,
    slate_pl_profile_generator,)
# for each settings file, generate a bloc slate profile using slate pl 

NUM_REPS = 10 # number of preference profile samples to generate per district
district_nums = [d_config["num_districts"] for d_config in districting_config]
# for each districting plan and each district of that plan, generate NUM_REPS preference profiles and save them to file
for rep in range(NUM_REPS):
    for num_districts in district_nums:
        if num_districts == 1:
            print("Warning: Only one district, skipping preference profile generation for now since no districting plan yet")
            break
        settings_folder = Path(f"outputs/{RUN_NAME}/settings/{num_districts}_districts")
        for settings_path in settings_folder.glob("*.json"): # loop through all plans and all districts 
            settings_file = open(settings_path, 'r')
            settings = json.load(settings_file)
            #print(settings_path)
            config = BlocSlateConfig(
                n_voters=settings["num_voters"],
                slate_to_candidates=settings["slate_to_candidates"],
                bloc_proportions=settings["bloc_proportions"],
                cohesion_mapping=settings["cohesion_parameters"],

            )
            setting_file_stem = Path(settings_path).stem
            profile_folder = Path(f"outputs/{RUN_NAME}/profiles/{num_districts}_districts")
            profile_folder.mkdir(parents=True, exist_ok=True)
            profile_output = profile_folder / f"{setting_file_stem.replace('settings', 'profile')}_rep_v{rep+1}.csv"
            config.set_dirichlet_alphas(settings["alphas"])
            profile = slate_pl_profile_generator(config)
            profile.to_csv(profile_output)
            settings_file.close()

/var/folders/8p/z1k7jgpx2dj57tsnh25vy08c0000gn/T/ipykernel_4850/4112989623.py:17: ResourceWarning: unclosed file <_io.TextIOWrapper name='outputs/attempt_2/settings/6_districts/attempt_2_6_districts_settings_sample_191_district_1.json' mode='r' encoding='UTF-8'>
  settings_file = open(settings_path, 'r')


In [ ]:
def get_winners(elected_candidates):
    winners = []
    for candidate in elected_candidates:
        winners.append(str(next(iter(candidate))))
    return winners
# simulate elections for each preference profile and save results to file
election_results_folder = Path(f"outputs/{RUN_NAME}/election_results")
election_results_folder.mkdir(parents=True, exist_ok=True)
from votekit import RankProfile
from votekit.elections import Plurality, FastSTV as STV, Borda, Cumulative, RankedPairs
for dc in districting_config:
    print(dc["num_districts"])
    if dc["num_districts"] == 3 or dc["num_districts"] == 1:
        continue
    profile_files = Path(f"outputs/{RUN_NAME}/profiles/{dc['num_districts']}_districts").glob("*.csv")
    all_election_results = []
    all_profile_files = []
    for num_profile, profile_file in enumerate(profile_files):
        pp = RankProfile.from_csv(profile_file)
        all_profile_files.append(str(profile_file))
        if dc["num_winners_per_district"] > 1:
            # stv, borda, cumulative, ranked pairs
            elected_stv = STV(pp, n_seats=dc["num_winners_per_district"], tiebreak='random').get_elected()
            elected_borda = Borda(pp, n_seats=dc["num_winners_per_district"], tiebreak='random').get_elected()
            elected_cumulative = Cumulative(pp, n_seats=dc["num_winners_per_district"], tiebreak='random').get_elected()
            elected_ranked_pairs = RankedPairs(pp, n_seats=dc["num_winners_per_district"], tiebreak='random').get_elected()
            election_results = {
                "stv": get_winners(elected_stv),
                "borda": get_winners(elected_borda),
                "cumulative": get_winners(elected_cumulative),
                "ranked_pairs": get_winners(elected_ranked_pairs)
            }
        else:
            # plurality, irv
            elected_plurality = Plurality(pp, n_seats=1, tiebreak='random').get_elected()
            elected_irv = STV(pp, n_seats=1, tiebreak='random').get_elected()
            print("election_results")
            print(elected_plurality)
            print(elected_irv)
            election_results = {"plurality": get_winners(elected_plurality), "irv": get_winners(elected_irv)}
            print(election_results)
        all_election_results.append(election_results)
    print(all_election_results)
    # save election results to file
    out_path = election_results_folder / f"{RUN_NAME}_{dc['num_districts']}_districts_{dc['num_winners_per_district']}_winners_election_results.json"
    json_info = {
        "run_name" : RUN_NAME,
        "num_districts": dc["num_districts"],
        "num_winners_per_district": dc["num_winners_per_district"],
        "election_results": all_election_results,
        "profile_file_list" : all_profile_files
    }
    print("json info and outpath")
    print(json_info)
    print(out_path)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(json_info, f, indent=4)
        break

6
election_results
(frozenset({'B3'}),)
(frozenset({'B3'}),)
{'plurality': ['B3'], 'irv': ['B3']}
election_results
(frozenset({'NB3'}),)
(frozenset({'NB3'}),)
{'plurality': ['NB3'], 'irv': ['NB3']}
[{'plurality': ['B3'], 'irv': ['B3']}, {'plurality': ['NB3'], 'irv': ['NB3']}]
json info and outpath
{'run_name': 'attempt_2', 'num_districts': 6, 'num_winners_per_district': 1, 'election_results': [{'plurality': ['B3'], 'irv': ['B3']}, {'plurality': ['NB3'], 'irv': ['NB3']}], 'profile_file_list': ['outputs/attempt_2/profiles/6_districts/attempt_2_6_districts_profile_sample_466_district_4_rep_v2.csv', 'outputs/attempt_2/profiles/6_districts/attempt_2_6_districts_profile_sample_467_district_5_rep_v7.csv', 'outputs/attempt_2/profiles/6_districts/attempt_2_6_districts_profile_sample_740_district_3_rep_v6.csv']}
outputs/attempt_2/election_results/attempt_2_6_districts_1_winners_election_results.json


In [110]:
print(all_election_results)

[{'plurality': ['B3'], 'irv': ['B3']}, {'plurality': ['NB3'], 'irv': ['NB3']}, {'plurality': ['NB2'], 'irv': ['NB2']}, {'plurality': ['B3'], 'irv': ['B3']}, {'plurality': ['B1'], 'irv': ['NB3']}, {'plurality': ['B2'], 'irv': ['NB2']}, {'plurality': ['NB3'], 'irv': ['NB3']}, {'plurality': ['NB2'], 'irv': ['NB2']}, {'plurality': ['B1'], 'irv': ['NB1']}, {'plurality': ['B2'], 'irv': ['NB1']}, {'plurality': ['NB2'], 'irv': ['NB2']}, {'plurality': ['NB2'], 'irv': ['NB2']}, {'plurality': ['NB3'], 'irv': ['NB3']}, {'plurality': ['NB1'], 'irv': ['NB1']}, {'plurality': ['NB1'], 'irv': ['NB1']}, {'plurality': ['NB3'], 'irv': ['NB3']}, {'plurality': ['B1'], 'irv': ['B1']}, {'plurality': ['NB3'], 'irv': ['NB3']}, {'plurality': ['NB1'], 'irv': ['NB1']}, {'plurality': ['B3'], 'irv': ['NB2']}, {'plurality': ['B2'], 'irv': ['NB3']}, {'plurality': ['NB1'], 'irv': ['NB1']}, {'plurality': ['NB2'], 'irv': ['NB1']}, {'plurality': ['B3'], 'irv': ['NB2']}, {'plurality': ['NB2'], 'irv': ['B2']}, {'plurality':

In [ ]:
# count number of seats won by slate B 
import re 
run_name = "attempt_6"
election_results_path = Path(f"outputs/{run_name}/election_results")
for generative_mode in ["name-cumulative", "slate-pl"]:
    election_results_files = Path(f"{election_results_path}/{generative_mode}").glob("*.json")
    for filepath in election_results_files:
        fname = filepath.stem
        match = re.search(r'_(\d+)_districts_(\d+)_winners', fname)
        num_districts = int(match.group(1))
        num_winners = int(match.group(2))
        # open file 
        # count the number of seats won by B candidate per each election 
        with open(filepath) as f:
            results = json.load(f)
            

3 2
1 6
6 1
3 2
1 6
6 1
